In [6]:
# ── Cài nếu chưa có ──────────────────────────────────────
# !pip install crawl4ai
# !crawl4ai-setup

import asyncio
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig
from crawl4ai.content_filter_strategy import PruningContentFilter
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

URL = "https://www.coinstrat.com/institutions/"

async def crawl(url: str):
    browser_cfg = BrowserConfig(headless=True, verbose=False)

    md_generator = DefaultMarkdownGenerator(
        content_filter=PruningContentFilter(threshold=0.48, threshold_type="fixed"),
    )
    run_cfg = CrawlerRunConfig(
        markdown_generator=md_generator,
        wait_until="networkidle",   # chờ JS render xong
        page_timeout=30_000,
        cache_mode="bypass",
    )

    async with AsyncWebCrawler(config=browser_cfg) as crawler:
        result = await crawler.arun(url=url, config=run_cfg)
    return result

result = await crawl(URL)
print(f"Success : {result.success}")
print(f"Status  : {result.status_code}")
print(f"URL     : {result.url}")

[FETCH]... ↓ https://www.coinstrat.com/institutions/                                                              | ✓ | ⏱: 7.80s 

[SCRAPE].. ◆ https://www.coinstrat.com/institutions/                                                              | ✓ | ⏱: 0.04s 

[COMPLETE] ● https://www.coinstrat.com/institutions/                                                              | ✓ | ⏱: 7.86s 

Success : True
Status  : 200
URL     : https://www.coinstrat.com/institutions/


In [7]:
# ── fit_markdown (đã lọc noise) ───────────────────────────
fit = result.markdown.fit_markdown or ""
print(f"fit_markdown length : {len(fit)} chars")
print("=" * 60)
print(fit[:3000])

fit_markdown length : 2469 chars
Coinstrat x Coinstore Partnership: Đánh Dấu Cột mốc Niêm Yết Mới[Đọc thêm →](https://www.coinstrat.com/blog/all/coinstrat-x-coinstore-partnership/)
VI
|
EN
Open menu
Sản Phẩm
[Doanh nghiệp](https://www.coinstrat.com/institutions/)[Mời bạn bè](https://www.coinstrat.com/refer-a-friend/)
Trung tâm hỗ trợ
[Blog](https://www.coinstrat.com/blog/)
Về chúng tôi
# Giải pháp tài chính cho các tổ chức
CoinStrat liên kết đến thị trường tài sản số toàn cầu, cho phép các nhà đầu tư truyền thống vay tài sản kỹ thuật số hoặc USD theo các điều khoản và tỷ lệ thương lượng để bảo hiểm rủi ro, tạo thị trường, bán khống hoặc cho các nhu cầu vốn lưu động khác. Chúng tôi thực hiện các hoạt động như ký quỹ, bán khống, cung cấp báo cáo và có thể tùy chỉnh các sản phẩm cho vay của mình cho chiến lược kinh doanh cụ thể của từng khách hàng. Nền tảng tài chính của chúng tôi hỗ trợ các khoản vay bằng stablecoin và tiền điện tử vốn hóa lớn, chẳng hạn như bitcoin, ether và các loại ti

In [8]:
# ── raw_markdown (toàn bộ, chưa lọc) ─────────────────────
raw = result.markdown.raw_markdown or ""
print(f"raw_markdown length : {len(raw)} chars")
print("=" * 60)
print(raw[:3000])

raw_markdown length : 5000 chars
Coinstrat x Coinstore Partnership: Đánh Dấu Cột mốc Niêm Yết Mới[Đọc thêm →](https://www.coinstrat.com/blog/all/coinstrat-x-coinstore-partnership/)
VI
|
EN
[![](data:image/svg+xml,%3csvg%20xmlns=%27http://www.w3.org/2000/svg%27%20version=%271.1%27%20width=%27180%27%20height=%2738.88%27/%3e)![coinstrat](https://www.coinstrat.com/_next/image/?url=%2Flogo.png&w=384&q=75)](https://www.coinstrat.com/)
Open menu
Sản Phẩm
[Doanh nghiệp](https://www.coinstrat.com/institutions/)[Mời bạn bè](https://www.coinstrat.com/refer-a-friend/)
Trung tâm hỗ trợ
[Blog](https://www.coinstrat.com/blog/)
Về chúng tôi
[Đăng nhập](https://app.coinstrat.com/auth/login)[Đăng kí](https://app.coinstrat.com/auth/register)
![](https://images.prismic.io/coinstrat-web/ee4ad22b-29f0-46e7-b3f5-f4f833715471_photo-1525130413817-d45c1d127c42.jpeg?auto=compress,format)
# Giải pháp tài chính cho các tổ chức
CoinStrat liên kết đến thị trường tài sản số toàn cầu, cho phép các nhà đầu tư truyền th

In [9]:
# ── Metadata & links ──────────────────────────────────────
print("Title   :", result.metadata.get("title"))
print("Desc    :", result.metadata.get("description"))
print(f"Links   : {len(result.links.get('internal', []))} internal  |  {len(result.links.get('external', []))} external")
print("\nInternal links (top 10):")
for lnk in result.links.get("internal", [])[:10]:
    print(" ", lnk.get("href"))

Title   : Giải Pháp Tài Chính Cho Các Tổ Chức - CoinStrat
Desc    : None
Links   : 20 internal  |  5 external

Internal links (top 10):
  https://www.coinstrat.com/blog/all/coinstrat-x-coinstore-partnership/
  https://www.coinstrat.com/
  https://www.coinstrat.com/institutions/
  https://www.coinstrat.com/refer-a-friend/
  https://www.coinstrat.com/blog/
  https://app.coinstrat.com/auth/login
  https://app.coinstrat.com/auth/register
  https://www.coinstrat.com/crypto-interest-account
  https://www.coinstrat.com/crypto-loans
  https://www.coinstrat.com/trading


In [4]:
# ── So sánh độ dài fit vs raw ─────────────────────────────
fit_len = len(fit)
raw_len = len(raw)
ratio   = fit_len / raw_len * 100 if raw_len else 0
print(f"raw_markdown : {raw_len:,} chars")
print(f"fit_markdown : {fit_len:,} chars  ({ratio:.1f}% của raw)")
print(f"Đã lọc bỏ   : {raw_len - fit_len:,} chars noise")

NameError: name 'raw' is not defined